In [40]:
%%capture
!pip install transformers datasets evaluate rouge_score jieba accelerate -q

In [41]:

import os
import math
import warnings
import numpy as np
import torch
import jieba
import evaluate

from dataclasses import dataclass
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    set_seed,
)

warnings.filterwarnings('ignore')
set_seed(42)

# ── 设备检测 ──────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU : {gpu_name}  |  VRAM : {vram_gb:.1f} GB')
else:
    print('⚠️  CPU 模式（训练会较慢）')

✅ GPU : Tesla T4  |  VRAM : 15.6 GB


In [42]:
@dataclass
class CFG:
    # ── 模型 & 数据 ──────────────────────────────────────
    model_name        : str   = 'fnlp/bart-base-chinese'
    dataset_name      : str   = 'hugcyp/LCSTS'
    output_dir        : str   = './bart-chinese-summarization'

    # ── 数据处理 ─────────────────────────────────────────
    max_input_length  : int   = 512    # 原文最大 token 数
    max_target_length : int   = 64     # 摘要最大 token 数
    train_size        : int   = 50000  # 训练集样本数 (None = 全量)
    val_size          : int   = 3000   # 验证集样本数
    test_size         : int   = 3000   # 测试集样本数

    # ── 训练超参 ─────────────────────────────────────────
    num_epochs        : int   = 5
    train_batch_size  : int   = 16     # T4/P100 适配 (显存不足时改 8)
    eval_batch_size   : int   = 32
    grad_accum_steps  : int   = 2      # 等效 batch_size = 32
    learning_rate     : float = 3e-5
    weight_decay      : float = 0.01
    warmup_ratio      : float = 0.05
    fp16              : bool  = True   # P100 上若报错改 False
    label_smoothing   : float = 0.1

    # ── 生成超参 ─────────────────────────────────────────
    num_beams         : int   = 4
    no_repeat_ngram   : int   = 2

    # ── 日志 & 保存 ──────────────────────────────────────
    logging_steps     : int   = 50
    eval_steps        : int   = 500
    save_steps        : int   = 500
    save_total_limit  : int   = 2
    early_stop_patience: int  = 3

cfg = CFG()
print('📋 配置加载完成')
print(f'   等效 batch size : {cfg.train_batch_size * cfg.grad_accum_steps}')
print(f'   FP16            : {cfg.fp16}')

📋 配置加载完成
   等效 batch size : 32
   FP16            : True


In [43]:
print('📦 加载 Tokenizer ...')
tokenizer = BertTokenizer.from_pretrained(cfg.model_name)

print('📦 加载模型 ...')
model = BartForConditionalGeneration.from_pretrained(cfg.model_name)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'✅ 模型加载完成')
print(f'   总参数量      : {total_params:.1f}M')
print(f'   可训练参数量  : {train_params:.1f}M')

📦 加载 Tokenizer ...
📦 加载模型 ...
✅ 模型加载完成
   总参数量      : 140.2M
   可训练参数量  : 140.2M


In [44]:
print('📥 加载 LCSTS 数据集 ...')
raw_ds = load_dataset(cfg.dataset_name)
print(raw_ds)

# ── 列名探测（兼容不同版本的 LCSTS 列名）────────────────
sample = raw_ds['train'][0]
print('\n样本字段:', list(sample.keys()))
print('样本内容:', sample)

📥 加载 LCSTS 数据集 ...
DatasetDict({
    train: Dataset({
        features: ['summary', 'text'],
        num_rows: 2400591
    })
    validation: Dataset({
        features: ['summary', 'text'],
        num_rows: 8685
    })
    test: Dataset({
        features: ['summary', 'text'],
        num_rows: 725
    })
})

样本字段: ['summary', 'text']
样本内容: {'summary': '修改后的立法法全文公布', 'text': '新华社受权于18日全文播发修改后的《中华人民共和国立法法》，修改后的立法法分为“总则”“法律”“行政法规”“地方性法规、自治条例和单行条例、规章”“适用与备案审查”“附则”等6章，共计105条。'}


In [45]:
# ── 自动识别文本列 & 摘要列 ───────────────────────────────
def detect_columns(sample: dict):
    """从样本字段名推断 (text_col, summary_col)"""
    keys = list(sample.keys())
    # 常见列名映射
    text_candidates    = ['text', 'content', 'article', 'passage', 'src']
    summary_candidates = ['summary', 'title', 'headline', 'abstract', 'tgt']
    text_col    = next((k for k in keys if k.lower() in text_candidates), keys[0])
    summary_col = next((k for k in keys if k.lower() in summary_candidates), keys[1])
    return text_col, summary_col

TEXT_COL, SUMMARY_COL = detect_columns(sample)
print(f'✅ 文本列   : {TEXT_COL}')
print(f'✅ 摘要列   : {SUMMARY_COL}')

✅ 文本列   : text
✅ 摘要列   : summary


In [46]:
def preprocess(examples):
    """Tokenize 原文与摘要，生成 labels"""
    inputs = tokenizer(
        examples[TEXT_COL],
        max_length=cfg.max_input_length,
        truncation=True,
        padding=False,          # 由 DataCollator 动态 padding
    )
    with tokenizer.as_target_tokenizer():
        targets = tokenizer(
            examples[SUMMARY_COL],
            max_length=cfg.max_target_length,
            truncation=True,
            padding=False,
        )
    inputs['labels'] = targets['input_ids']
    return inputs


# ── 裁剪数据量（加速 Kaggle 调试） ────────────────────────
def slice_split(split_name, size):
    ds = raw_ds[split_name]
    if size and size < len(ds):
        ds = ds.select(range(size))
    return ds

train_raw = slice_split('train',      cfg.train_size)
val_raw   = slice_split('validation', cfg.val_size)
test_raw  = slice_split('test',       cfg.test_size)

print(f'训练集 : {len(train_raw):,} 条')
print(f'验证集 : {len(val_raw):,} 条')
print(f'测试集 : {len(test_raw):,} 条')

print('\n⚙️  预处理数据集 ...')
tokenize_kwargs = dict(
    batched=True,
    remove_columns=train_raw.column_names,
    num_proc=2,
    desc='Tokenizing',
)
train_ds = train_raw.map(preprocess, **tokenize_kwargs)
val_ds   = val_raw.map(preprocess,   **tokenize_kwargs)
test_ds  = test_raw.map(preprocess,  **tokenize_kwargs)

print('✅ 预处理完成')
print(train_ds)

训练集 : 50,000 条
验证集 : 3,000 条
测试集 : 725 条

⚙️  预处理数据集 ...
✅ 预处理完成
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 50000
})


In [47]:
rouge_metric = evaluate.load('rouge')

def tokenize_zh(text: str) -> str:
    """中文按字符切分（jieba 分词后用空格连接，供 ROUGE 计算）"""
    return ' '.join(jieba.cut(text))


def compute_metrics(eval_pred):
    preds, labels = eval_pred

    # 解码预测序列
    if isinstance(preds, tuple):
        preds = preds[0]
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # 中文分词处理
    decoded_preds  = [tokenize_zh(p.strip()) for p in decoded_preds]
    decoded_labels = [tokenize_zh(l.strip()) for l in decoded_labels]

    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False,
    )
    # 保留 4 位小数
    result = {k: round(v * 100, 4) for k, v in result.items()}

    # 平均生成长度
    gen_len = np.mean([np.count_nonzero(p) for p in preds])
    result['gen_len'] = round(gen_len, 2)
    return result


print('✅ 评估函数就绪（ROUGE-1 / ROUGE-2 / ROUGE-L）')

✅ 评估函数就绪（ROUGE-1 / ROUGE-2 / ROUGE-L）


In [48]:
## 6.5 自定义训练日志 Callback
from transformers import TrainerCallback
import time

class PrintLossCallback(TrainerCallback):
    """每 logging_steps 步强制 print loss 到 Kaggle Cell 输出"""

    def __init__(self):
        self._t0 = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or not state.is_local_process_zero:
            return
        step    = state.global_step
        elapsed = time.time() - self._t0
        loss    = logs.get("loss",            "--")
        lr      = logs.get("learning_rate",   "--")
        epoch   = logs.get("epoch",           "--")
        # 训练 loss
        if "loss" in logs:
            print(
                f"[Step {step:>6}]  "
                f"loss={loss:.4f}  "
                f"lr={lr:.2e}  "
                f"epoch={epoch:.2f}  "
                f"elapsed={elapsed/60:.1f}min",
                flush=True,
            )
        # 验证集结果
        if "eval_rougeL" in logs:
            r1 = logs.get("eval_rouge1",  0)
            r2 = logs.get("eval_rouge2",  0)
            rl = logs.get("eval_rougeL",  0)
            el = logs.get("eval_loss",    0)
            print(
                f"[Eval  {step:>6}]  "
                f"loss={el:.4f}  "
                f"R1={r1:.2f}  R2={r2:.2f}  RL={rl:.2f}",
                flush=True,
            )

print("✅ PrintLossCallback 就绪")


✅ PrintLossCallback 就绪


In [49]:
# ── Data Collator（动态 padding + label padding -100）────
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if cfg.fp16 else None,
)

# ── 训练参数 ─────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=cfg.output_dir,

    # 训练
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.train_batch_size,
    per_device_eval_batch_size=cfg.eval_batch_size,
    gradient_accumulation_steps=cfg.grad_accum_steps,
    learning_rate=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    warmup_ratio=cfg.warmup_ratio,
    lr_scheduler_type='cosine',
    fp16=cfg.fp16,
    label_smoothing_factor=cfg.label_smoothing,

    # 生成
    predict_with_generate=True,
    generation_max_length=cfg.max_target_length,
    generation_num_beams=cfg.num_beams,

    # 评估 & 保存
    eval_strategy='steps',
    eval_steps=cfg.eval_steps,
    save_strategy='steps',
    save_steps=cfg.save_steps,
    save_total_limit=cfg.save_total_limit,
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,

    # 日志
    logging_dir=f'{cfg.output_dir}/logs',
    logging_steps=cfg.logging_steps,
    logging_strategy="steps",
    logging_first_step=True,
    report_to='none',           # 改成 'tensorboard' 可开启 TB 记录

    # 其它
    dataloader_num_workers=2,
    seed=42,
    optim='adamw_torch',
)

# ── Trainer ──────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg.early_stop_patience),PrintLossCallback()],
)

print('✅ Trainer 初始化完成')

✅ Trainer 初始化完成


In [50]:
print('🚀 开始训练...')
train_result = trainer.train()

# ── 保存最优模型 ─────────────────────────────────────────
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

# ── 训练指标 ─────────────────────────────────────────────
metrics = train_result.metrics
metrics['train_samples'] = len(train_ds)
trainer.log_metrics('train', metrics)
trainer.save_metrics('train', metrics)
trainer.save_state()

print('\n✅ 训练完成！')
print(f'   训练时长  : {metrics["train_runtime"]:.0f} 秒')
print(f'   样本/秒   : {metrics["train_samples_per_second"]:.2f}')
print(f'   最终 Loss : {metrics["train_loss"]:.4f}')

🚀 开始训练...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
500,6.925400,3.620380,9.776500,0.811100,9.686100,9.705600,17.670000
1000,6.571100,3.577137,10.680500,0.992900,10.580600,10.601100,17.300000
1500,6.512400,3.548465,10.607000,1.146700,10.500700,10.513700,16.190000
2000,6.200500,3.540366,10.968400,1.114400,10.869700,10.868200,16.720000
2500,6.090500,3.535535,10.887700,1.156000,10.784500,10.791100,17.140000
3000,6.107800,3.531773,11.298500,1.074500,11.187700,11.200600,17.000000
3500,5.954800,3.535323,11.313600,1.189300,11.233300,11.216900,16.970000


[Step      1]  loss=12.2877  lr=1.53e-07  epoch=0.00  elapsed=0.0min
[Step     50]  loss=9.9798  lr=7.65e-06  epoch=0.06  elapsed=0.8min
[Step    100]  loss=7.7466  lr=1.53e-05  epoch=0.13  elapsed=1.6min
[Step    150]  loss=7.4447  lr=2.30e-05  epoch=0.19  elapsed=2.4min
[Step    200]  loss=7.3258  lr=3.00e-05  epoch=0.26  elapsed=3.1min
[Step    250]  loss=7.2239  lr=3.00e-05  epoch=0.32  elapsed=3.9min
[Step    300]  loss=7.1265  lr=2.99e-05  epoch=0.38  elapsed=4.7min
[Step    350]  loss=7.0496  lr=2.99e-05  epoch=0.45  elapsed=5.5min
[Step    400]  loss=6.9752  lr=2.98e-05  epoch=0.51  elapsed=6.3min
[Step    450]  loss=6.9676  lr=2.97e-05  epoch=0.58  elapsed=7.1min
[Step    500]  loss=6.9254  lr=2.95e-05  epoch=0.64  elapsed=7.9min


Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.621 seconds.
DEBUG:jieba:Loading model cost 0.621 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


[Eval     500]  loss=3.6204  R1=9.78  R2=0.81  RL=9.69
[Step    550]  loss=6.8980  lr=2.93e-05  epoch=0.70  elapsed=11.4min
[Step    600]  loss=6.8621  lr=2.91e-05  epoch=0.77  elapsed=12.2min
[Step    650]  loss=6.8716  lr=2.89e-05  epoch=0.83  elapsed=12.9min
[Step    700]  loss=6.8594  lr=2.87e-05  epoch=0.90  elapsed=13.7min
[Step    750]  loss=6.8084  lr=2.84e-05  epoch=0.96  elapsed=14.5min
[Step    800]  loss=6.7559  lr=2.81e-05  epoch=1.02  elapsed=15.3min
[Step    850]  loss=6.5882  lr=2.78e-05  epoch=1.09  elapsed=16.1min
[Step    900]  loss=6.6151  lr=2.74e-05  epoch=1.15  elapsed=16.8min
[Step    950]  loss=6.5722  lr=2.70e-05  epoch=1.22  elapsed=17.6min
[Step   1000]  loss=6.5711  lr=2.67e-05  epoch=1.28  elapsed=18.4min
[Eval    1000]  loss=3.5771  R1=10.68  R2=0.99  RL=10.58
[Step   1050]  loss=6.5563  lr=2.62e-05  epoch=1.34  elapsed=21.7min
[Step   1100]  loss=6.5817  lr=2.58e-05  epoch=1.41  elapsed=22.5min
[Step   1150]  loss=6.5560  lr=2.54e-05  epoch=1.47  elapsed

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


***** train metrics *****
  epoch                    =     4.9968
  total_flos               = 16848587GF
  train_loss               =     6.4346
  train_runtime            = 1:18:11.13
  train_samples            =      50000
  train_samples_per_second =     53.292
  train_steps_per_second   =      0.832

✅ 训练完成！
   训练时长  : 4691 秒
   样本/秒   : 53.29
   最终 Loss : 6.4346


In [51]:
print('📊 验证集评估...')
val_metrics = trainer.evaluate(eval_dataset=val_ds, metric_key_prefix='val')
trainer.log_metrics('val', val_metrics)
trainer.save_metrics('val', val_metrics)

print('\n📈 验证集指标:')
for k, v in val_metrics.items():
    if any(k.startswith(p) for p in ['val_rouge', 'val_loss', 'val_gen']):
        print(f'   {k:<30} : {v}')

📊 验证集评估...


early stopping required metric_for_best_model, but did not find eval_rougeL so early stopping is disabled


***** val metrics *****
  epoch                  =     4.9968
  val_gen_len            =      16.97
  val_loss               =     3.5353
  val_rouge1             =    11.3136
  val_rouge2             =     1.1893
  val_rougeL             =    11.2333
  val_rougeLsum          =    11.2169
  val_runtime            = 0:02:24.74
  val_samples_per_second =     20.725
  val_steps_per_second   =      0.325

📈 验证集指标:
   val_loss                       : 3.535322904586792
   val_rouge1                     : 11.3136
   val_rouge2                     : 1.1893
   val_rougeL                     : 11.2333
   val_rougeLsum                  : 11.2169
   val_gen_len                    : 16.97


In [52]:
print('🧪 测试集评估...')
test_metrics = trainer.predict(
    test_dataset=test_ds,
    metric_key_prefix='test',
    num_beams=cfg.num_beams,
    max_length=cfg.max_target_length,
    no_repeat_ngram_size=cfg.no_repeat_ngram,
).metrics

trainer.log_metrics('test', test_metrics)
trainer.save_metrics('test', test_metrics)

print('\n📈 测试集指标:')
for k, v in test_metrics.items():
    if any(k.startswith(p) for p in ['test_rouge', 'test_loss', 'test_gen']):
        print(f'   {k:<30} : {v}')

🧪 测试集评估...


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

***** test metrics *****
  test_gen_len            =      17.21
  test_loss               =     5.8636
  test_rouge1             =        0.0
  test_rouge2             =        0.0
  test_rougeL             =        0.0
  test_rougeLsum          =        0.0
  test_runtime            = 0:00:34.31
  test_samples_per_second =      21.13
  test_steps_per_second   =       0.35

📈 测试集指标:
   test_loss                      : 5.86359167098999
   test_rouge1                    : 0.0
   test_rouge2                    : 0.0
   test_rougeL                    : 0.0
   test_rougeLsum                 : 0.0
   test_gen_len                   : 17.21


In [54]:
def generate_summary(text: str, model=model, tokenizer=tokenizer) -> str:
    """对单条文本生成摘要"""
    model.eval()
    inputs = tokenizer(
        text,
        max_length=cfg.max_input_length,
        truncation=True,
        return_tensors='pt',
        return_token_type_ids=False,
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            num_beams=cfg.num_beams,
            max_length=cfg.max_target_length,
            no_repeat_ngram_size=cfg.no_repeat_ngram,
            early_stopping=True,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# ── 随机取 5 条测试集样本展示效果 ────────────────────────
print('=' * 70)
print('🔍 推理示例（测试集随机 5 条）')
print('=' * 70)

indices = np.random.choice(len(test_raw), 5, replace=False)
for i, idx in enumerate(indices, 1):
    sample    = test_raw[int(idx)]
    src_text  = sample[TEXT_COL]
    ref_text  = sample[SUMMARY_COL]
    gen_text  = generate_summary(src_text)

    print(f'\n【样本 {i}】')
    print(f'原文  : {src_text[:100]}...')
    print(f'参考  : {ref_text}')
    print(f'生成  : {gen_text}')
    print('-' * 70)

🔍 推理示例（测试集随机 5 条）

【样本 1】
原文  : 英国超六成的餐饮连锁店提供的冰块细菌总量，甚至超过取自马桶水箱的水。中国相关快餐连锁店表示，制冰用的是过滤纯净水，应该没恩替。中国农业大学副教授朱毅称，食物多样化，品牌尽量多样化是规避风险的最好办法。...
参考  : 
生成  : 英 国 超 六 成 快 餐 店 提 供 冰 块 细 菌 总 量 超 过 马 桶 水
----------------------------------------------------------------------

【样本 2】
原文  : 去年年底以来，全球最大的几只对冲基金只把赌注下在了一个地方——日本。他们这次大规模做空日元的活动，被业内称为安倍交易。这场战役的最大赢家，就是曾经狙击英镑，被称为“打败英国央行的人”----索罗斯。短...
参考  : 
生成  : 索 罗 斯 狙 击 英 镑 的 最 大 赢 家
----------------------------------------------------------------------

【样本 3】
原文  : 宏观政策如何出牌，就像打麻将。庄家胡什么花色，不能由着自己性子。做不了大牌，小胡也行，但绝不能点炮。眼下就是政策决断时刻。经济整体下行，消费出不了多大力，投资和出口都下行，三驾马车，两匹马打出溜，一匹...
参考  : 
生成  : 宏 观 政 策 如 何 出 牌
----------------------------------------------------------------------

【样本 4】
原文  : 土耳其航空公司6月9日从伊斯坦布尔飞往北京的TK020航班，由于ATC(航空管制)的原因，导致在伊斯坦布尔机场延误5小时。飞机到达北京后，旅客要求乘务长及机长给予解释，却被侮辱为劫机。@土耳其航空公司...
参考  : 
生成  : 土 耳 其 航 班 在 伊 斯 坦 布 尔 延 误 5 小 时
----------------------------------------------------------------------

【样本 5】
原文  : 此前网传广东高考作文题为《另一面》，该消息不实。今年广东高考作文材料是历史学家汤因比及居里夫

In [55]:
import shutil

# 压缩微调后模型，方便从 Kaggle 下载
archive_path = shutil.make_archive(
    base_name='bart-chinese-summarization',
    format='zip',
    root_dir='.',
    base_dir=cfg.output_dir,
)
print(f'✅ 模型已压缩至: {archive_path}')
print(f'   文件大小    : {os.path.getsize(archive_path)/1e6:.1f} MB')

✅ 模型已压缩至: /kaggle/working/bart-chinese-summarization.zip
   文件大小    : 3583.5 MB
